<div dir="rtl">

# الدرس 57: تقييم أنظمة النماذج اللغوية

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### البيانات، ونظام البحث

</div>

In [ ]:
import re
import math
import zlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    ("hours", "ساعات الدوام", "تستقبل العيادة المرضى من السبت إلى الخميس، من التاسعة صباحاً حتى التاسعة مساءً. يوم الجمعة عطلة، ما عدا قسم الطوارئ."),
    ("emergency", "الطوارئ", "قسم الطوارئ مفتوح على مدار الساعة، في كل أيام الأسبوع، ولا يحتاج موعداً مسبقاً. الحالات الخطيرة لها الأولوية دائماً."),
    ("booking", "حجز المواعيد", "احجز موعدك عبر الموقع الإلكتروني، أو بالاتصال بالرقم 4455 في أوقات الدوام. تُفتح المواعيد قبل أسبوعين."),
    ("cancel", "إلغاء المواعيد", "يمكن إلغاء الموعد أو تغييره مجاناً قبل 24 ساعة على الأقل. الإلغاء المتأخر أو عدم الحضور عليه رسوم 50 ليرة."),
    ("prices", "الأسعار", "المعاينة عند طبيب عام 400 ليرة، وعند طبيب مختص 700 ليرة. المراجعة خلال أسبوعين من المعاينة مجانية."),
    ("insurance", "التأمين", "نقبل التأمين الحكومي وأغلب شركات التأمين الخاصة. أحضر بطاقة التأمين وهويتك الشخصية عند التسجيل."),
    ("payment", "الدفع", "الدفع نقداً أو ببطاقة مصرفية أو ائتمانية. لا نقبل الشيكات."),
    ("location", "العنوان", "العيادة في شارع الورد رقم 12، قرب محطة الترام. يوجد موقف سيارات مجاني خلف المبنى."),
    ("pediatrics", "طب الأطفال", "قسم الأطفال فيه طبيبتان، ويستقبل من الأحد إلى الخميس في الفترة الصباحية."),
    ("lab", "المختبر", "المختبر يفتح من الثامنة صباحاً. تحليل السكر الصائم يحتاج صياماً ثماني ساعات قبل سحب الدم."),
    ("results", "نتائج التحاليل", "النتائج جاهزة عادة خلال يوم عمل واحد. تصلك رسالة نصية، ويمكنك تحميل النتيجة من الموقع برقم ملفك."),
    ("vaccines", "اللقاحات", "نوفر لقاحات الأطفال حسب الجدول الوطني، ولقاح الإنفلونزا الموسمي ابتداء من شهر أكتوبر."),
    ("dental", "الأسنان", "قسم الأسنان يعمل مساءً فقط، من الرابعة حتى التاسعة. تنظيف الأسنان وتبييضها يحتاجان موعداً مسبقاً."),
    ("imaging", "الأشعة", "نوفر الأشعة السينية والتصوير بالموجات فوق الصوتية. الرنين المغناطيسي غير متوفر، ونحوّلك إلى مركز شريك."),
    ("prescriptions", "الوصفات", "لتجديد وصفة دواء دائم، اطلبها عبر الموقع إذا كانت الوصفة السابقة أحدث من ستة أشهر، وإلا فاحجز مراجعة قصيرة."),
    ("telehealth", "الاستشارة عن بعد", "استشارة الفيديو متوفرة للمراجعات فقط، وليس للمعاينة الأولى. سعرها 300 ليرة."),
    ("reports", "التقارير الطبية", "يُسلَّم التقرير الطبي خلال ثلاثة أيام عمل من طلبه، ورسومه 150 ليرة."),
    ("privacy", "الخصوصية", "ملفك الطبي سري. لا نشارك معلوماتك مع أي جهة إلا بموافقتك المكتوبة، أو بطلب قانوني رسمي."),
    ("companions", "المرافقون", "يُسمح بمرافق واحد مع كل مريض بالغ، وبمرافقَين مع الأطفال."),
    ("accessibility", "سهولة الوصول", "في المبنى مصعد ومدخل مخصص للكراسي المتحركة، وأماكن وقوف قريبة من الباب لذوي الإعاقة."),
    ("languages", "اللغات", "أطباؤنا يتحدثون العربية والتركية والإنجليزية."),
    ("nutrition", "التغذية", "أخصائية التغذية تستقبل يومي الاثنين والأربعاء. الجلسة الأولى 500 ليرة، والمتابعة 300 ليرة."),
    ("complaints", "الشكاوى", "لتقديم شكوى أو اقتراح، استخدم النموذج على الموقع، أو اطلب مسؤول خدمة المرضى في الاستقبال."),
    ("pharmacy", "الصيدلية", "لا توجد صيدلية داخل العيادة. أقرب صيدلية على بعد مئة متر، مقابل محطة الترام."),
]
train_q = [
    ("متى تفتحون أبوابكم؟", "hours"), ("ما هي أوقات العمل؟", "hours"), ("هل أنتم مفتوحون يوم الجمعة؟", "hours"),
    ("عندي حالة خطيرة الآن، هل يمكنني المجيء؟", "emergency"), ("هل الإسعاف يعمل في الليل؟", "emergency"), ("أحتاج علاجاً عاجلاً بعد منتصف الليل", "emergency"),
    ("كيف أحجز عند الطبيب؟", "booking"), ("ما رقم الهاتف لأخذ موعد؟", "booking"), ("هل يمكن الحجز أونلاين؟", "booking"),
    ("أريد أن ألغي موعدي", "cancel"), ("هل أدفع غرامة إذا لم أحضر؟", "cancel"), ("كيف أغير وقت الموعد؟", "cancel"),
    ("كم تكلفة الكشف؟", "prices"), ("بكم زيارة الطبيب الأخصائي؟", "prices"), ("هل المراجعة بعد الزيارة مدفوعة؟", "prices"),
    ("هل تقبلون بطاقة التأمين الصحي؟", "insurance"), ("عندي تأمين خاص، هل يغطي الزيارة؟", "insurance"), ("ما الأوراق المطلوبة للتأمين؟", "insurance"),
    ("هل أستطيع الدفع بالفيزا؟", "payment"), ("هل تأخذون شيكات؟", "payment"), ("طرق الدفع المتاحة؟", "payment"),
    ("أين مكان العيادة بالضبط؟", "location"), ("هل يوجد مكان لركن السيارة؟", "location"), ("كيف أصل إليكم بالمواصلات؟", "location"),
    ("ابني مريض، هل عندكم طبيب أطفال؟", "pediatrics"), ("متى يداوم قسم الأطفال؟", "pediatrics"), ("أبحث عن دكتورة أطفال", "pediatrics"),
    ("هل يجب أن أصوم قبل التحليل؟", "lab"), ("متى يفتح المختبر؟", "lab"), ("أريد فحص سكر في الدم", "lab"),
    ("متى تطلع نتيجة التحليل؟", "results"), ("كيف أستلم نتائج الفحص؟", "results"), ("هل ترسلون النتيجة على الجوال؟", "results"),
    ("هل عندكم تطعيمات للأطفال؟", "vaccines"), ("متى يتوفر مطعوم الانفلونزا؟", "vaccines"), ("أريد تطعيم ابنتي", "vaccines"),
    ("عندي ألم في ضرسي", "dental"), ("متى يعمل طبيب الأسنان؟", "dental"), ("أريد تبييض أسناني", "dental"),
    ("هل تجرون صورة رنين؟", "imaging"), ("أحتاج صورة أشعة لركبتي", "imaging"), ("هل عندكم إيكو؟", "imaging"),
    ("دوائي انتهى وأريد وصفة جديدة", "prescriptions"), ("كيف أجدد الروشتة؟", "prescriptions"), ("أحتاج نفس الدواء مرة ثانية", "prescriptions"),
    ("هل يمكن أن أكلم الطبيب بالفيديو؟", "telehealth"), ("هل توجد استشارة أونلاين؟", "telehealth"), ("بكم الاستشارة عبر الإنترنت؟", "telehealth"),
    ("أحتاج تقريراً لعملي", "reports"), ("كم يستغرق إصدار التقرير الطبي؟", "reports"), ("هل التقرير الطبي مجاني؟", "reports"),
    ("هل معلوماتي الطبية محمية؟", "privacy"), ("هل ترسلون ملفي لأي أحد؟", "privacy"), ("من يستطيع رؤية سجلي الطبي؟", "privacy"),
    ("هل يمكن أن يدخل معي أحد؟", "companions"), ("كم شخصاً يرافق الطفل؟", "companions"), ("أمي تريد أن تأتي معي", "companions"),
    ("هل المبنى مناسب للكرسي المتحرك؟", "accessibility"), ("هل يوجد مصعد؟", "accessibility"), ("والدي لا يستطيع صعود الدرج", "accessibility"),
    ("هل الطبيب يتكلم إنجليزي؟", "languages"), ("لا أتكلم العربية جيداً، هل تفهمون التركية؟", "languages"), ("بأي لغة يتحدث الأطباء؟", "languages"),
    ("أريد برنامجاً لإنقاص الوزن", "nutrition"), ("هل عندكم أخصائي حمية؟", "nutrition"), ("بكم جلسة التغذية؟", "nutrition"),
    ("أريد أن أشتكي من سوء المعاملة", "complaints"), ("كيف أرسل ملاحظة عن الخدمة؟", "complaints"), ("لمن أتوجه باقتراح؟", "complaints"),
    ("من أين أشتري الدواء؟", "pharmacy"), ("هل توجد صيدلية قريبة؟", "pharmacy"), ("هل أجد الأدوية عندكم؟", "pharmacy"),
]
test_set = [
    ("ما مواعيد الدوام اليومية؟", "hours", "من التاسعة صباحاً حتى التاسعة مساءً"),
    ("هل تعملون يوم السبت؟", "hours", "من السبت إلى الخميس"),
    ("ابني ابتلع شيئاً، أين أذهب فوراً؟", "emergency", "قسم الطوارئ مفتوح على مدار الساعة"),
    ("أريد موعداً مع الطبيب الأسبوع القادم", "booking", "بالاتصال بالرقم 4455"),
    ("لن أستطيع الحضور غداً، ماذا أفعل؟", "cancel", "قبل 24 ساعة على الأقل"),
    ("كم أدفع مقابل زيارة الطبيب العام؟", "prices", "400 ليرة"),
    ("هل تغطي شركة التأمين تكاليف العلاج؟", "insurance", "أغلب شركات التأمين الخاصة"),
    ("هل يمكن الدفع بالكاش؟", "payment", "الدفع نقداً"),
    ("ما عنوانكم؟", "location", "شارع الورد رقم 12"),
    ("طفلي عنده حرارة، من يعالجه؟", "pediatrics", "قسم الأطفال فيه طبيبتان"),
    ("كم ساعة أصوم قبل فحص الدم؟", "lab", "ثماني ساعات"),
    ("وصلت نتيجة الفحص؟ كيف أراها؟", "results", "تحميل النتيجة من الموقع برقم ملفك"),
    ("هل لقاح الأنفلونزا متوفر عندكم؟", "vaccines", "ابتداء من شهر أكتوبر"),
    ("أريد تنظيف الأسنان", "dental", "من الرابعة حتى التاسعة"),
    ("طلب مني الطبيب رنيناً مغناطيسياً", "imaging", "الرنين المغناطيسي غير متوفر"),
    ("أريد تجديد وصفة الضغط", "prescriptions", "اطلبها عبر الموقع"),
    ("هل يمكن المعاينة الأولى عن بعد؟", "telehealth", "للمراجعات فقط"),
    ("أحتاج شهادة مرضية للجامعة", "reports", "خلال ثلاثة أيام عمل"),
    ("هل تشاركون بياناتي مع شركتي؟", "privacy", "إلا بموافقتك المكتوبة"),
    ("هل يستطيع زوجي مرافقتي؟", "companions", "بمرافق واحد مع كل مريض بالغ"),
    ("أستخدم كرسياً متحركاً، هل أستطيع الدخول؟", "accessibility", "مدخل مخصص للكراسي المتحركة"),
    ("Do you speak English?", "languages", "العربية والتركية والإنجليزية"),
    ("أحتاج أخصائية تغذية", "nutrition", "يومي الاثنين والأربعاء"),
    ("عندي شكوى على موظف الاستقبال", "complaints", "مسؤول خدمة المرضى"),
    ("هل أشتري العلاج من داخل العيادة؟", "pharmacy", "لا توجد صيدلية داخل العيادة"),
]
test_q = [(q, page) for q, page, _ in test_set]
answers = {q: a for q, _, a in test_set}
ids = [d[0] for d in docs]
index = {d: i for i, d in enumerate(ids)}
sections = [f"{title}. {body}" for _, title, body in docs]
page_text = dict(zip(ids, sections))

def normalize(text):
    text = re.sub(r"[ً-ْـ]", "", text)
    text = re.sub("[إأآ]", "ا", text)
    return text.replace("ة", "ه").replace("ى", "ي").lower()

def light_stem(w):
    for p in ("وال", "بال", "كال", "فال", "لل", "ال"):
        if w.startswith(p) and len(w) - len(p) >= 2:
            w = w[len(p):]
            break
    for _ in range(2):
        for s in ("ات", "ون", "ين", "ها", "هم", "كم", "ني", "ه", "ي", "ك", "ا"):
            if w.endswith(s) and len(w) - len(s) >= 3:
                w = w[:-len(s)]
                break
    return w

def terms(text):
    return [light_stem(w) for w in re.findall(r"\w+", normalize(text))]

class BM25:
    def __init__(self, texts, k1=1.5, b=0.75):
        self.docs = [Counter(terms(t)) for t in texts]
        self.lens = [sum(d.values()) for d in self.docs]
        self.avg = sum(self.lens) / len(self.lens)
        self.k1, self.b = k1, b
        N = len(texts)
        df = Counter(w for d in self.docs for w in d)
        self.idf = {w: math.log((N - n + 0.5) / (n + 0.5) + 1) for w, n in df.items()}

    def scores(self, query):
        q = terms(query)
        out = np.zeros(len(self.docs))
        for i, (tf, L) in enumerate(zip(self.docs, self.lens)):
            for w in q:
                if w in tf:
                    f = tf[w]
                    out[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * L / self.avg))
        return out

def char_retriever(chunks):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    m = vec.fit_transform(chunks)
    return lambda q: (vec.transform([q]) @ m.T).toarray()[0]

BUCKETS = 2 ** 15

def pieces(text):
    out = []
    for w in re.findall(r"\w+", text.lower()):
        w = f"<{w}>"
        for n in (2, 3, 4):
            out += [w[i:i + n] for i in range(len(w) - n + 1)]
    return out

class TinyEncoder(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.table = nn.EmbeddingBag(BUCKETS, dim, mode="mean")
        nn.init.normal_(self.table.weight, std=0.1)

    def forward(self, batch):
        flat, starts = [], []
        for text in batch:
            starts.append(len(flat))
            flat += [zlib.crc32(p.encode()) % BUCKETS for p in pieces(text)]
        return F.normalize(self.table(torch.tensor(flat), torch.tensor(starts)), dim=-1)

torch.manual_seed(0)
rng = np.random.default_rng(0)
encoder = TinyEncoder()
questions_of = {}
for question, page in train_q:
    questions_of.setdefault(page, []).append(question)
opt = torch.optim.Adam(encoder.parameters(), lr=0.02)
for step in range(201):
    chosen = rng.choice(len(docs), size=16, replace=False)
    Q = encoder([rng.choice(questions_of[ids[j]]) for j in chosen])
    D = encoder([sections[j] for j in chosen])
    loss = F.cross_entropy(Q @ D.T / 0.1, torch.arange(16))
    opt.zero_grad()
    loss.backward()
    opt.step()
with torch.no_grad():
    section_vecs = encoder(sections)

def encoder_scores(question):
    with torch.no_grad():
        return (section_vecs @ encoder([question])[0]).numpy()

def rrf(*score_fns, k=60):
    def fused(question):
        total = np.zeros(len(sections))
        for fn in score_fns:
            order = np.argsort(-fn(question), kind="stable")
            total[order] += 1 / (k + np.arange(1, len(order) + 1))
        return total
    return fused

bm25 = BM25(sections)
char_scores = char_retriever(sections)
hybrid = rrf(bm25.scores, char_scores, encoder_scores)
print("ready:", len(sections), "sections,", len(test_set), "test questions")

<div dir="rtl">

### مقاييس البحث

</div>

In [ ]:
def retrieval_report(score_fn, questions=test_q, cutoff=10):
    ranks = []
    for q, gold in questions:
        order = np.argsort(-score_fn(q), kind="stable")
        ranks.append(int(np.where(order == index[gold])[0][0]) + 1)
    ranks = np.array(ranks)
    rr = np.where(ranks <= cutoff, 1 / ranks, 0.0)           # reciprocal rank, zero beyond the cutoff
    report = {f"recall@{k}": round(float((ranks <= k).mean()), 2) for k in (1, 3, 5)}
    report["MRR"] = round(float(rr.mean()), 3)
    return report

for name, fn in [("BM25", bm25.scores), ("tiny encoder", encoder_scores), ("hybrid", hybrid)]:
    print(f"{name:13}", retrieval_report(fn))

<div dir="rtl">

### نظام مرجعي بسيط

</div>

In [ ]:
def clauses(section):
    body = section.split(". ", 1)[1]                            # drop the title
    return [c.strip() for c in re.split(r"[.،؟]", body) if len(c.split()) >= 2]

def extractive_answer(question):
    page = int(np.argmax(hybrid(question)))
    candidates = clauses(sections[page])
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit(candidates + [question])
    sims = (vec.transform([question]) @ vec.transform(candidates).T).toarray()[0]
    return candidates[int(np.argmax(sims))]

for q in ["كم أدفع مقابل زيارة الطبيب العام؟", "هل تعملون يوم السبت؟"]:
    print(q, "->", extractive_answer(q))

<div dir="rtl">

### المطابقة والتداخل

</div>

In [ ]:
def answer_tokens(text):
    text = re.sub(r"\[\d+\]", " ", text)                        # citation numbers are not part of the answer
    text = re.sub(r"[^\w\s]", " ", normalize(text))             # punctuation
    out = []
    for w in text.split():
        for p in ("وال", "بال", "لل", "ال"):
            if w.startswith(p) and len(w) - len(p) >= 2:
                w = w[len(p):]
                break
        out.append(w)
    return out

def exact_match(pred, gold):
    return float(answer_tokens(pred) == answer_tokens(gold))

def token_f1(pred, gold):
    p, g = answer_tokens(pred), answer_tokens(gold)
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(p), common / len(g)
    return 2 * precision * recall / (precision + recall)

def contains(pred, gold):
    p, g = answer_tokens(pred), answer_tokens(gold)
    return float(any(p[i:i + len(g)] == g for i in range(len(p) - len(g) + 1)))

print(answer_tokens("المعاينةُ عند الطبيب العامّ: 400 ليرة [1]."))
print("F1 of the hand example:", token_f1("المعاينة 400 ليرة", "400 ليرة"), token_f1("450 ليرة", "400 ليرة"))

preds = {q: extractive_answer(q) for q, _, _ in test_set}
for name, metric in [("exact match", exact_match), ("token F1", token_f1), ("contains the reference", contains)]:
    print(f"{name:23} {np.mean([metric(preds[q], a) for q, _, a in test_set]):.2f}")

<div dir="rtl">

### سجل أجوبة من نموذج لغوي

</div>

In [ ]:
log = [
    ("كم أدفع مقابل زيارة الطبيب العام؟", "المعاينة عند الطبيب العام 400 ليرة [1].", ["prices", "pharmacy", "booking"], 1),
    ("ما مواعيد الدوام اليومية؟", "نستقبلكم من السبت إلى الخميس، من التاسعة صباحاً حتى التاسعة مساءً [1].", ["hours", "booking", "dental"], 1),
    ("كم ساعة أصوم قبل فحص الدم؟", "يجب الصيام اثنتي عشرة ساعة قبل التحليل [1].", ["lab", "results", "booking"], 0),
    ("هل يمكن المعاينة الأولى عن بعد؟", "نعم، يمكنك إجراء المعاينة الأولى بالفيديو، وسعرها 300 ليرة [1].", ["telehealth", "prices", "booking"], 0),
    ("ما عنوانكم؟", "العيادة في شارع الورد رقم 12، قرب محطة الترام [1].", ["location", "hours", "pharmacy"], 1),
    ("هل يستطيع زوجي مرافقتي؟", "نعم، يُسمح بمرافق واحد مع كل مريض بالغ [2].", ["privacy", "companions", "accessibility"], 1),
    ("أحتاج شهادة مرضية للجامعة", "لا أعرف. اتصل بالعيادة على الرقم 4455.", ["prescriptions", "hours", "imaging"], 0),
    ("هل لقاح الأنفلونزا متوفر عندكم؟", "نعم، لقاح الإنفلونزا الموسمي متوفر ابتداء من شهر أكتوبر [1].", ["vaccines", "imaging", "telehealth"], 1),
    ("هل تشاركون بياناتي مع شركتي؟", "لا، لا نشارك معلوماتك مع أي جهة إلا بموافقتك المكتوبة [1].", ["privacy", "companions", "imaging"], 1),
    ("أريد تنظيف الأسنان", "قسم الأسنان يعمل صباحاً من التاسعة، والتنظيف يحتاج موعداً مسبقاً [1].", ["dental", "imaging", "pediatrics"], 0),
    ("أحتاج أخصائية تغذية", "أخصائية التغذية تستقبل يومي الاثنين والأربعاء، والجلسة الأولى 450 ليرة [1].", ["nutrition", "prescriptions", "cancel"], 0),
]

def number_problems(answer, sources):
    """the check from lesson 56: every number in the answer must be in a cited source"""
    if answer.strip().startswith("لا أعرف"):
        return []
    cited = {int(n) for n in re.findall(r"\[(\d+)\]", answer)}
    allowed = set(re.findall(r"\d+", " ".join(page_text[sources[n - 1]] for n in cited if 1 <= n <= len(sources))))
    return [n for n in re.findall(r"\d+", re.sub(r"\[\d+\]", "", answer)) if n not in allowed]

human = np.array([label for *_, label in log])
f1s = np.array([token_f1(ans, answers[q]) for q, ans, _, _ in log])
cont = np.array([contains(ans, answers[q]) for q, ans, _, _ in log])
rule = np.array([0 if number_problems(ans, src) else 1 for q, ans, src, _ in log])
print(" human  F1   contains  numbers-ok  question")
for (q, *_), hu, f, c, r in zip(log, human, f1s, cont, rule):
    print(f"   {hu}   {f:.2f}     {c:.0f}         {r}       {q}")

<div dir="rtl">

### كم يتفق كل مقياس مع البشر؟

</div>

In [ ]:
def cohen_kappa(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    po = (a == b).mean()                                        # observed agreement
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())  # agreement expected by chance
    return (po - pe) / (1 - pe)

auto = {"F1 >= 0.5": (f1s >= 0.5).astype(int), "contains the reference": cont.astype(int), "numbers ok": rule}
for name, pred in auto.items():
    print(f"{name:23} agreement {np.mean(pred == human):.2f}   kappa {cohen_kappa(pred, human):.2f}")

<div dir="rtl">

### الحكَم: الطلب والتحليل

</div>

In [ ]:
%pip install -q -U anthropic

In [ ]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
import anthropic

class FakeAPI(BaseHTTPRequestHandler):
    replies = []
    received = []

    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        FakeAPI.received.append(body)
        status, reply = FakeAPI.replies.pop(0)
        data = json.dumps(reply).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, *args):
        pass

server = HTTPServer(("127.0.0.1", 0), FakeAPI)
threading.Thread(target=server.serve_forever, daemon=True).start()

def fake_text_reply(text, n_in, n_out):
    return 200, {"id": "msg_fake", "type": "message", "role": "assistant", "model": "fake",
                 "content": [{"type": "text", "text": text}], "stop_reason": "end_turn",
                 "stop_sequence": None, "usage": {"input_tokens": n_in, "output_tokens": n_out}}

client = anthropic.Anthropic(api_key="fake-key", base_url=f"http://127.0.0.1:{server.server_port}", max_retries=0)

In [ ]:
JUDGE_SYSTEM = """أنت مقيّم صارم لأجوبة مساعد عيادة.
ستحصل على: السؤال، والمصادر، والجواب المرجعي، وجواب المساعد.
الجواب صحيح فقط إذا اتفق مع المصادر في كل معلومة فيه، وأجاب عن السؤال.
الصياغة المختلفة لا تهم إذا كان المعنى نفسه.
اكتب السبب أولاً، ثم الحكم. وأرجع كائن جيسون فقط:
{"reason": "...", "verdict": "correct"} أو {"reason": "...", "verdict": "incorrect"}"""

def judge_messages(question, answer, sources):
    src = "\n".join(f'<source id="{n}">{page_text[p]}</source>' for n, p in enumerate(sources, 1))
    return [{"role": "user", "content": f"<question>{question}</question>\n<sources>\n{src}\n</sources>\n"
                                        f"<reference>{answers[question]}</reference>\n<answer>{answer}</answer>"}]

def parse_verdict(text):
    match = re.search(r"\{.*\}", text, flags=re.S)             # the model may add words around the JSON
    try:
        verdict = json.loads(match.group(0))["verdict"]
    except (AttributeError, ValueError, KeyError):
        return None                                              # unparseable: count it, never guess
    return {"correct": 1, "incorrect": 0}.get(str(verdict).lower())

VERDICT_SCHEMA = {"type": "object",
                  "properties": {"reason": {"type": "string"}, "verdict": {"type": "string", "enum": ["correct", "incorrect"]}},
                  "required": ["reason", "verdict"], "additionalProperties": False}

def judge(question, answer, sources):
    reply = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=300, system=JUDGE_SYSTEM,
                                   messages=judge_messages(question, answer, sources),
                                   output_config={"format": {"type": "json_schema", "schema": VERDICT_SCHEMA}})
    return parse_verdict(reply.content[0].text)

canned = [
    '{"reason": "المصدر يقول إن الاستشارة عن بعد للمراجعات فقط، والجواب يقول إن المعاينة الأولى ممكنة.", "verdict": "incorrect"}',
    'بالتأكيد! هذا تقييمي:\n{"reason": "الجواب يطابق ساعات الدوام في المصدر.", "verdict": "correct"}',
    '{"reason": "السعر في المصدر 500، وليس 450", verdict: incorrect}',
]
for (q, ans, src, hu), text in zip([log[3], log[1], log[10]], canned):
    FakeAPI.replies.append(fake_text_reply(text, 350, 40))
    print(f"judge: {judge(q, ans, src)}   human: {hu}   | {q}")
print("\noutput format sent:", json.dumps(FakeAPI.received[-1]["output_config"], ensure_ascii=False)[:90], "...")

<div dir="rtl">

## التقييم كاختبارات

</div>

In [ ]:
def test_retrieval_quality():
    r = retrieval_report(hybrid)
    assert r["recall@3"] >= 0.85 and r["MRR"] >= 0.80, r

def test_no_invented_numbers():
    bad = [q for q, ans, src, _ in log if number_problems(ans, src)]
    assert not bad, f"numbers not found in the sources: {bad}"

for test in (test_retrieval_quality, test_no_invented_numbers):
    try:
        test()
        print("PASS", test.__name__)
    except AssertionError as e:
        print("FAIL", test.__name__, "->", e)

<div dir="rtl">

## تمرين: أي حكَم تختار؟

جرّبت حكَمين على الأجوبة العشرة نفسها التي حكم عليها إنسان.

الحكَم الأول:

| | الإنسان: صحيح | الإنسان: خطأ |
|---|---|---|
| الحكَم: صحيح | 6 | 1 |
| الحكَم: خطأ | 0 | 3 |

والحكَم الثاني كسول: يقول "صحيح" دائماً.

1. احسب بيدك نسبة الاتفاق وكابا لكل حكَم.
2. أي حكَم تختار؟ وما الخطر الوحيد الباقي في الحكَم الأول؟
3. تحقق بالكود، بدالة كابا من الدرس.

</div>

In [ ]:
# اكتب حلّك هنا
